In [1]:
!rm -rf /kaggle/working/*

In [2]:
import copy
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision.models import resnet18
from tqdm import tqdm

In [3]:
# ──────────────────────────────────────────────────────────────────────────────
# 1. Data helpers
# ──────────────────────────────────────────────────────────────────────────────
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2023, 0.1994, 0.2010)

def get_cifar10(data_dir, train=True):
    transform = T.Compose([
        T.ToTensor(),
        T.Normalize(CIFAR10_MEAN, CIFAR10_STD),
    ])
    return torchvision.datasets.CIFAR10(
        root=data_dir, train=train, download=True, transform=transform
    )


def class_indices(dataset, class_id):
    """Return indices of all samples belonging to class_id."""
    targets = np.array(dataset.targets)
    return np.where(targets == class_id)[0].tolist()


def denormalize(tensor):
    """Convert normalized tensor back to [0,1] for visualisation / PGD."""
    mean = torch.tensor(CIFAR10_MEAN, device=tensor.device).view(3,1,1)
    std  = torch.tensor(CIFAR10_STD,  device=tensor.device).view(3,1,1)
    return tensor * std + mean


def renormalize(tensor):
    """Re-normalise a [0,1] tensor back to CIFAR-10 space."""
    mean = torch.tensor(CIFAR10_MEAN, device=tensor.device).view(3,1,1)
    std  = torch.tensor(CIFAR10_STD,  device=tensor.device).view(3,1,1)
    return (tensor - mean) / std


class PoisonedDataset(Dataset):
    """
    Wraps a base-class dataset and replaces a subset of images with
    their poisoned counterparts x̃_b, while keeping the original label.
    """
    def __init__(self, base_dataset, base_class_indices, poisoned_images, poison_indices):
        self.dataset         = base_dataset
        self.base_idx        = base_class_indices   # all base-class sample indices
        self.poisoned_images = poisoned_images       # dict: base_class_idx → poisoned tensor
        self.poison_indices  = set(poison_indices)  # which base_class indices are poisoned

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        if idx in self.poison_indices:
            img = self.poisoned_images[idx]
        return img, label


In [4]:
# ──────────────────────────────────────────────────────────────────────────────
# 2. Feature extractor wrapper
# ──────────────────────────────────────────────────────────────────────────────

class FeatureExtractor(nn.Module):
    """
    ResNet18 with the final FC replaced by a task-specific head.
    Exposes the penultimate 512-d embedding via .embed(x).
    """
    def __init__(self, num_classes=10, pretrained=True):
        super().__init__()
        weights = torchvision.models.ResNet18_Weights.DEFAULT if pretrained else None
        base = resnet18(weights=weights)
        # Keep everything up to the avg-pool
        self.backbone = nn.Sequential(*list(base.children())[:-1])  # → (B, 512, 1, 1)
        self.fc       = nn.Linear(512, num_classes)

    def embed(self, x):
        """Return penultimate 512-d feature vector."""
        z = self.backbone(x)          # (B, 512, 1, 1)
        return z.flatten(1)           # (B, 512)

    def forward(self, x):
        return self.fc(self.embed(x))



In [5]:
# ──────────────────────────────────────────────────────────────────────────────
# 3. Phase 1 — Surrogate training & prototype extraction
# ──────────────────────────────────────────────────────────────────────────────

def train_surrogate(model, train_dataset, target_class, epochs, batch_size, device):
    """
    Fine-tune the surrogate model on target-class images only.
    This specialises the penultimate layer to discriminative target features.
    """
    print(f"\n[Phase 1] Fine-tuning surrogate on class {target_class} only...")

    idx     = class_indices(train_dataset, target_class)
    subset  = Subset(train_dataset, idx)
    loader  = DataLoader(subset, batch_size=batch_size, shuffle=True, num_workers=2)

    model.to(device)
    opt     = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
    crit    = nn.CrossEntropyLoss()

    for ep in range(epochs):
        model.train()
        total_loss = 0.0
        for imgs, labels in tqdm(loader, desc=f"  Surrogate epoch {ep+1}/{epochs}", leave=False):
            imgs, labels = imgs.to(device), labels.to(device)
            opt.zero_grad()
            loss = crit(model(imgs), labels)
            loss.backward()
            opt.step()
            total_loss += loss.item()
        print(f"  epoch {ep+1:2d}  loss={total_loss/len(loader):.4f}")

    return model


def extract_prototype(model, train_dataset, target_class, batch_size, device):
    """
    Compute p_t = mean embedding of all target-class training images.
    This is the centroid of the target class in feature space.
    """
    print(f"\n[Phase 1] Extracting target-class prototype...")
    idx    = class_indices(train_dataset, target_class)
    subset = Subset(train_dataset, idx)
    loader = DataLoader(subset, batch_size=batch_size, shuffle=False, num_workers=2)

    model.eval()
    embeddings = []
    with torch.no_grad():
        for imgs, _ in tqdm(loader, desc="  Encoding target class", leave=False):
            z = model.embed(imgs.to(device))
            embeddings.append(z.cpu())

    prototype = torch.cat(embeddings, dim=0).mean(0)  # (512,)
    print(f"  prototype shape: {prototype.shape}, norm: {prototype.norm():.3f}")
    return prototype


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
print(f"Using device: {device}")

Using device: cuda


In [7]:
target_class = 3  # cat
base_class   = 5  # dog
CIFAR10_CLASSES = ["airplane","automobile","bird","cat","deer",
                   "dog","frog","horse","ship","truck"]
print(f"Target class: {target_class} ({CIFAR10_CLASSES[target_class]})")
print(f"Base   class: {base_class}   ({CIFAR10_CLASSES[base_class]})")


Target class: 3 (cat)
Base   class: 5   (dog)


In [8]:
# ── Load data ─────────────────────────────────────────────────────────────
data_dir = "./data"
train_dataset = get_cifar10(data_dir, train=True)
test_dataset  = get_cifar10(data_dir, train=False)

100%|██████████| 170M/170M [00:02<00:00, 77.2MB/s] 


In [9]:
# ── Phase 1: Surrogate training ───────────────────────────────────────────
surrogate = FeatureExtractor(num_classes=10, pretrained=True)
surrogate_epochs = 15
surrogate = train_surrogate(
    surrogate, train_dataset, target_class,
    surrogate_epochs, batch_size=32, device=device)

torch.save(surrogate.state_dict(), "surrogate_model.pth")



[Phase 1] Fine-tuning surrogate on class 3 only...


  epoch  1  loss=0.2962


  epoch  2  loss=0.0152


  epoch  3  loss=0.0067


  epoch  4  loss=0.0039


...

  epoch 11  loss=0.0007


  epoch 12  loss=0.0006


  epoch 13  loss=0.0005


  epoch 14  loss=0.0005


  epoch 15  loss=0.0004


In [10]:
# # Recreate model architecture from saved model
# surrogate = FeatureExtractor(num_classes=10, pretrained=True)

# # Load saved weights
# surrogate.load_state_dict(torch.load("surrogate_model.pth", map_location=device))

# # Move to device
# surrogate.to(device)

# print("Model loaded.")

In [11]:
prototype = extract_prototype(
    surrogate, train_dataset, target_class, batch_size=32, device=device)



[Phase 1] Extracting target-class prototype...


  prototype shape: torch.Size([512]), norm: 21.097


In [12]:
# ──────────────────────────────────────────────────────────────────────────────
# 4. Phase 2a — Latent interpolation
# ──────────────────────────────────────────────────────────────────────────────

def interpolate_latent(z_b, p_t, lam):
    """
    z̃ = (1-λ)·z_b + λ·p_t
    Moves the base embedding toward the target prototype by λ.
    """
    return (1 - lam) * z_b + lam * p_t


In [13]:
# ──────────────────────────────────────────────────────────────────────────────
# 4. Phase 2b — Feature inversion via PGD
# ──────────────────────────────────────────────────────────────────────────────
def feature_inversion_pgd(model, x_b, z_target, epsilon, pgd_steps, pgd_lr, device):
    """
    Find x̃_b ≈ argmin ‖f(x') - z_target‖₂  s.t.  ‖x' - x_b‖_∞ ≤ ε

    We optimise in raw pixel space [0,1] (denormalised), clip to the ε-ball,
    then renormalise before each forward pass.

    Args:
        model     : surrogate FeatureExtractor (eval mode)
        x_b       : normalised input tensor (1, 3, H, W)
        z_target  : target embedding (1, 512)
        epsilon   : L-inf budget in [0,1] pixel space
        pgd_steps : number of gradient steps
        pgd_lr    : step size

    Returns:
        x̃_b : poisoned image tensor in normalised space (1, 3, H, W)
    """
    model.eval()
    x_b       = x_b.to(device)
    z_target  = z_target.to(device)

    # Work in [0,1] pixel space for clipping
    x_orig_01 = denormalize(x_b).detach().clone()
    x_adv_01  = x_orig_01.clone().requires_grad_(True)

    for step in range(pgd_steps):
        # Re-normalise → forward → embedding loss
        x_norm = renormalize(x_adv_01.clamp(0, 1))
        z_curr = model.embed(x_norm)
        loss   = torch.norm(z_curr - z_target, p=2)

        loss.backward()

        with torch.no_grad():
            # Gradient descent step
            grad = x_adv_01.grad

            #Normalize the gradient to have consistent step sizes across iterations
            # grad_normalized = grad / (torch.norm(grad, p=1) + 1e-8)
            grad_norm = torch.norm(grad.view(grad.shape[0], -1), dim=1)
            grad_norm = grad_norm.view(-1, 1, 1, 1) + 1e-8
            grad_normalized = grad / grad_norm

            x_adv_01 -= pgd_lr * grad_normalized
            # Project onto L-inf ε-ball around x_orig
            delta = (x_adv_01 - x_orig_01).clamp(-epsilon, epsilon)
            x_adv_01.copy_((x_orig_01 + delta).clamp(0, 1))

        x_adv_01.grad = None

    return renormalize(x_adv_01.detach().clamp(0, 1))

In [14]:
def stealthiness_check(x_orig, x_poison, epsilon, ssim_threshold=0.92):
    """
    Two stealthiness checks:
      1. L-inf distance in pixel space (hard constraint)
      2. SSIM (perceptual similarity, soft check)
    Returns (l_inf_ok, ssim_score).
    """
    orig_01   = denormalize(x_orig.cpu()).squeeze().permute(1,2,0).numpy()
    poison_01 = denormalize(x_poison.cpu()).squeeze().permute(1,2,0).numpy()
    orig_01   = np.clip(orig_01, 0, 1)
    poison_01 = np.clip(poison_01, 0, 1)

    l_inf = np.abs(orig_01 - poison_01).max()
    l_inf_ok = l_inf <= epsilon + 1e-4

    ssim_score = None
    if HAS_SKIMAGE:
        ssim_score = ssim(orig_01, poison_01, channel_axis=2, data_range=1.0)

    return l_inf_ok, ssim_score

In [15]:

# ──────────────────────────────────────────────────────────────────────────────
# 5. Phase 2c — Build poisoned dataset
# ──────────────────────────────────────────────────────────────────────────────

def build_poisoned_dataset(
    surrogate, train_dataset, base_class, prototype,
    lam, epsilon, poison_rate, pgd_steps, pgd_lr, batch_size, device
):
    """
    1. Select a random poison_rate fraction of base-class training images.
    2. For each: encode → interpolate → invert → verify stealthiness.
    3. Return a PoisonedDataset and metadata.
    """
    print(f"\n[Phase 2] Building poisoned dataset (λ={lam}, ε={epsilon:.4f}, rate={poison_rate})...")

    base_idx     = class_indices(train_dataset, base_class)
    n_poison     = int(len(base_idx) * poison_rate)
    poison_idx   = random.sample(base_idx, n_poison)

    print(f"  Base class samples : {len(base_idx)}")
    print(f"  Poisoning {n_poison} of them  ({poison_rate*100:.0f}%)")

    poisoned_images = {}
    p_t = prototype.unsqueeze(0).to(device)  # (1, 512)

    surrogate.eval()
    fail_count = 0

    for idx in tqdm(poison_idx, desc="  Inverting interpolated embeddings"):
        x_b, _ = train_dataset[idx]
        x_b     = x_b.unsqueeze(0).to(device)  # (1, 3, H, W)

        with torch.no_grad():
            z_b = surrogate.embed(x_b)          # (1, 512)

        z_tilde = interpolate_latent(z_b, p_t, lam)   # (1, 512)

        x_poison = feature_inversion_pgd(
            surrogate, x_b, z_tilde, epsilon, pgd_steps, pgd_lr, device
        )

        l_inf_ok, ssim_score = stealthiness_check(x_b, x_poison, epsilon)
        if not l_inf_ok:
            fail_count += 1

        poisoned_images[idx] = x_poison.squeeze(0).cpu()

    print(f"  Done. L-inf violations: {fail_count}/{n_poison}")
    if HAS_SKIMAGE:
        # Compute mean SSIM over a sample of 20 poisoned images
        sample = random.sample(list(poisoned_images.items()), min(20, len(poisoned_images)))
        ssim_scores = []
        for idx, x_p in sample:
            x_b, _ = train_dataset[idx]
            _, s = stealthiness_check(x_b.unsqueeze(0), x_p.unsqueeze(0), epsilon)
            ssim_scores.append(s)
        print(f"  Mean SSIM (sample of 20): {np.mean(ssim_scores):.4f}")

    return PoisonedDataset(train_dataset, base_idx, poisoned_images, poison_idx)


In [16]:
try:
    from skimage.metrics import structural_similarity as ssim
    HAS_SKIMAGE = True
except ImportError:
    HAS_SKIMAGE = False
    print("Warning: scikit-image not installed. SSIM disabled.")


In [17]:
# ── Phase 2: Poison generation ────────────────────────────────────────────
lambda_interp = 0.35    # interpolation factor
epsilon       = 8/255  # L-inf budget in pixel space
poison_rate   = 0.05   # fraction of base-class images to poison
pgd_steps    = 60
pgd_lr       = 0.02


poisoned_dataset = build_poisoned_dataset(
    surrogate, train_dataset, base_class, prototype,
    lam        = lambda_interp,
    epsilon    = epsilon,
    poison_rate= poison_rate,
    pgd_steps  = pgd_steps,
    pgd_lr     = pgd_lr,
    batch_size = 32,
    device     = device
)


[Phase 2] Building poisoned dataset (λ=0.35, ε=0.0314, rate=0.05)...
  Base class samples : 5000
  Poisoning 250 of them  (5%)


  Inverting interpolated embeddings: 100%|██████████| 250/250 [01:48<00:00,  2.30it/s]

  Done. L-inf violations: 0/250
  Mean SSIM (sample of 20): 0.9993


In [18]:
# Visualise some poison pairs
def visualize_poison_pairs(train_dataset, poisoned_dataset, poison_indices, out_dir, n=8):
    """
    Plot n original / poisoned image pairs side by side.
    Differences are shown in a 10× amplified difference map.
    """
    fig, axes = plt.subplots(3, n, figsize=(n * 2, 6))
    sample = random.sample(list(poison_indices), min(n, len(poison_indices)))

    for col, idx in enumerate(sample):
        x_orig, label = train_dataset[idx]
        x_pois        = poisoned_dataset.poisoned_images[idx]

        orig_np  = denormalize(x_orig.unsqueeze(0)).squeeze().permute(1,2,0).numpy().clip(0,1)
        pois_np  = denormalize(x_pois.unsqueeze(0)).squeeze().permute(1,2,0).numpy().clip(0,1)
        diff_np  = np.abs(orig_np - pois_np) * 10  # amplify diff

        axes[0, col].imshow(orig_np);   axes[0, col].axis("off")
        axes[1, col].imshow(pois_np);   axes[1, col].axis("off")
        axes[2, col].imshow(diff_np.clip(0,1), cmap="inferno"); axes[2, col].axis("off")

        if col == 0:
            axes[0, col].set_ylabel("Original", fontsize=9)
            axes[1, col].set_ylabel("Poisoned", fontsize=9)
            axes[2, col].set_ylabel("Diff ×10", fontsize=9)

    plt.suptitle(f"Class {label} ({CIFAR10_CLASSES[label]}) — poison pairs", fontsize=11)
    plt.tight_layout()
    out = Path(out_dir) / "poison_pairs.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {out}")

In [19]:
# Visualise some poison pairs
out_dir = "./outputs"
os.makedirs(out_dir, exist_ok=True)
visualize_poison_pairs(
    train_dataset, poisoned_dataset,
    list(poisoned_dataset.poison_indices), out_dir
)


  Saved: outputs/poison_pairs.png


In [20]:
# ──────────────────────────────────────────────────────────────────────────────
# 6. Phase 3 — Victim model fine-tuning
# ──────────────────────────────────────────────────────────────────────────────

def train_victim(model, poisoned_dataset, epochs, batch_size, device):
    """
    Fine-tune the victim ResNet18 on the full poisoned training set.
    The victim sees D_b (clean) ∪ D_p (poisoned, same labels).
    """
    print(f"\n[Phase 3] Fine-tuning victim model on poisoned dataset...")

    loader = DataLoader(poisoned_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    model.to(device)
    opt  = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit = nn.CrossEntropyLoss()

    for ep in range(epochs):
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        for imgs, labels in tqdm(loader, desc=f"  Victim epoch {ep+1}/{epochs}", leave=False):
            imgs, labels = imgs.to(device), labels.to(device)
            opt.zero_grad()
            out  = model(imgs)
            loss = crit(out, labels)
            loss.backward()
            opt.step()
            total_loss += loss.item()
            correct    += (out.argmax(1) == labels).sum().item()
            total      += labels.size(0)
        sched.step()
        print(f"  epoch {ep+1:2d}  loss={total_loss/len(loader):.4f}  train_acc={correct/total:.3f}")

    return model


In [21]:
# ── Phase 3: Victim training ──────────────────────────────────────────────
victim_epochs = 20
victim = FeatureExtractor(num_classes=10, pretrained=True)
victim = train_victim(victim, poisoned_dataset, victim_epochs, batch_size=32, device=device)


[Phase 3] Fine-tuning victim model on poisoned dataset...


  epoch  1  loss=1.4410  train_acc=0.518


  epoch  2  loss=1.0296  train_acc=0.650


  epoch  3  loss=0.8686  train_acc=0.702


  epoch  4  loss=0.7717  train_acc=0.737


...

  epoch 16  loss=0.0188  train_acc=0.994


  epoch 17  loss=0.0101  train_acc=0.997


  epoch 18  loss=0.0049  train_acc=0.999


  epoch 19  loss=0.0039  train_acc=0.999


  epoch 20  loss=0.0031  train_acc=0.999


In [22]:

# ──────────────────────────────────────────────────────────────────────────────
# 7. Phase 4 — Evaluation
# ──────────────────────────────────────────────────────────────────────────────
def extract_trigger(poisoned_dataset, train_dataset):
    """
    Extract a fixed trigger pattern from poisoned samples.
    """
    deltas = []

    for idx in poisoned_dataset.poison_indices:
        x_orig, _ = train_dataset[idx]
        x_poison = poisoned_dataset.poisoned_images[idx]

        x_orig = denormalize(x_orig).cpu().numpy()
        x_poison = denormalize(x_poison).cpu().numpy()

        delta = x_poison - x_orig
        deltas.append(delta)

    trigger = np.mean(deltas, axis=0)
    return torch.tensor(trigger, dtype=torch.float32)

def apply_trigger(x, trigger, epsilon=None):
    """
    Apply fixed learned trigger (no optimization).
    """
    x_01 = denormalize(x).clone()

    x_poison = x_01 + trigger

    if epsilon is not None:
        x_orig = x_01
        delta = torch.clamp(x_poison - x_orig, -epsilon, epsilon)
        x_poison = x_orig + delta

    x_poison = torch.clamp(x_poison, 0, 1)

    return renormalize(x_poison)

def evaluate(victim, test_dataset, trigger, target_class,
             batch_size, device, n_asr_samples=200):

    print("\n[Phase 4] Evaluation (FIXED BACKDOOR)")

    # ── CDA ─────────────────────────────
    loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    correct, total = 0, 0
    victim.eval()

    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)

            preds = victim(imgs).argmax(1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    cda = correct / total
    print(f"CDA = {cda:.4f}")

    # ── ASR ─────────────────────────────
    base_idx = class_indices(test_dataset, base_class)
    sample_idx = random.sample(base_idx, min(n_asr_samples, len(base_idx)))

    success = 0

    for idx in sample_idx:
        x, _ = test_dataset[idx]

        x_trigger = apply_trigger(x, trigger)

        with torch.no_grad():
            pred = victim(x_trigger.unsqueeze(0).to(device)).argmax(1).item()

        if pred == target_class:
            success += 1

    asr = success / len(sample_idx)

    print(f"ASR = {asr:.4f}")

    return cda, asr

In [23]:
  # ── Phase 4: Evaluation ───────────────────────────────────────────────────
trigger = extract_trigger(poisoned_dataset, train_dataset)
# ── Phase 4: Evaluation ─────────────────────────────────────────────
cda, asr = evaluate(
    victim,
    test_dataset,
    trigger,
    target_class,
    batch_size=32,
    device=device
)

print("\n" + "="*50)
print(f"  Final CDA : {cda:.4f}")
print(f"  Final ASR : {asr:.4f}")
print("="*50)


[Phase 4] Evaluation (FIXED BACKDOOR)
CDA = 0.8037
ASR = 0.1450

  Final CDA : 0.8037
  Final ASR : 0.1450


In [24]:
  # Save results
  result = {
      "target_class"  : target_class,
      "base_class"    : base_class,
      "lambda_interp" : lambda_interp,
      "epsilon"       : epsilon,
      "poison_rate"   : poison_rate,
      "cda"           : cda,
      "asr"           : asr,
  }
  import json
  with open(Path(out_dir) / "results.json", "w") as f:
      json.dump(result, f, indent=2)
  print(f"\n  Results saved to {out_dir}/results.json")




  Results saved to ./outputs/results.json
